In [ ]:
#0 Imports
#  
import numpy as np
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from scipy.signal import resample


In [ ]:
# 0 Load data as csv from MediaPipe Pose from database

import sqlite3
import pandas as pd

#Connect to database
db_path = "../landmark_database.db"
conn = sqlite3.connect(db_path)

#==========================================
#BASIC QUERIES
#==========================================
#View all data (limit to first 1000 rows)
df = pd.read_sql_query("SELECT * FROM landmarks LIMIT 100000", conn)
print(f"Total rows loaded: {len(df)}")
df.head()

Test_data = df.copy()

#Get specific patient data
#patient_df = pd.read_sql_query("""
#    SELECT * FROM landmarks 
#""", conn)

#df.head()

#Rows: 21,850,000 - 22,050,000

Total rows loaded: 100000


In [11]:
Test_data.tail(30)

,id,patient_name,frame,movement_type,jacket_status,side,model_name,timestamp_ms,landmark_id,x_norm,...,title,uploader,fps,start_time,end_time,duration,checksum,width,height,created_at
99970,99971,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,21,0.711586,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99971,99972,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,22,0.725127,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99972,99973,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,23,0.696310,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99973,99974,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,24,0.701167,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99974,99975,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,25,0.707770,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99975,99976,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,26,0.704635,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99976,99977,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,27,0.666555,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99977,99978,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,28,0.692548,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99978,99979,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,29,0.651192,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99979,99980,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,30,0.683132,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41


In [ ]:
# 1 Convert pose coordinates into pose column

import numpy as np
import pandas as pd

def add_pose_column(df):
    """
    Converts long-format MediaPipe dataframe into video-level dataframe
    with a pose column of shape (T, 33, 3).
    """

    pose_rows = []

    for source_file, group in df.groupby("source_file"):
        # Number of frames and joints
        T = group["frame"].nunique()
        J = 33  # MediaPipe Pose

        # Initialize pose array
        pose = np.zeros((T, J, 3), dtype=np.float32)

        # Fill pose array
        for _, r in group.iterrows():
            f = int(r.frame)
            j = int(r.landmark_id)
            pose[f, j, :] = [r.x_norm, r.y_norm, r.z_norm]

        # Keep one representative row for metadata
        row = group.iloc[0].copy()
        row["pose"] = pose

        pose_rows.append(row)

    return pd.DataFrame(pose_rows)

In [ ]:
# 2. Normalize the Pose

# Joint indices
LEFT_HEEL = 29
RIGHT_HEEL = 30
LEFT_HIP = 23
RIGHT_HIP = 24
LEFT_SHOULDER = 11
RIGHT_SHOULDER = 12

def normalize_pose_3d(pose):
    #pose: (T,J,3)
    pelvis = (pose[:, left_hip]+ pose[:, right_hip]) / 2

    #Translate: Express every joint position relative to the pelvis at each moment in time. Removing the pelvis position from all joints positions per frame centers the pose data dynamically. 
    # It remove the following effects: camera panning: distance-to.camera-effects, subject walking across the frame. 
    # What remains are relative joint motion, inter-limb coordination, postural deviations.
    pose_centered = pose - pelvis[:, None, :]
    # pose.shape == (T,J,3), pelvis.shape == (T,3). The above takes all frames, adds a new placeholder axis, then takes x,y coordinates -> (T,1,3).

    #Scale by torso length per frame
    # These joints are: more stable during gait, rarely occluded, Less noisy or affected by pathology than joints used for leg length measure,

    left_shoulder, right_shoulder = 11, 12
    torso = (pose_centered[:, left_shoulder] + pose_centered[:, right_shoulder]) / 2
    
    scale = np.linalg.norm(torso - pelvis, axis=1).mean() #calculates the torso length based on the average measures across time to get stable scale per clip.
    pose_scaled = pose_centered / scale # All coordinates are now expressed in units of torso_length, making them more comparable across subjects

    return pose_scaled

"""
#Removed:
✔ Body size differences
✔ Camera distance effects
Preserved:
✔ Relative joint motion
✔ Asymmetry
✔ Trunk lean & pelvic drop
✔ Temporal dynamics
"""


In [ ]:
#2. Sanity Checks:
# CHeck the hip centered trajectory oscillates around zeo
np.mean(pose_centered[:, left_hip], axis=0)
# should be close to [0, 0]

np.mean(np.linalg.norm(torso - pelvis, axis=1))
# should be ~1.0
    """
    Plot:
Knee trajectories from different subjects
They should be comparable in scale
    """
#check z coordinate is comparable to x,y, and tehre are no sudden jumps
plt.plot(pose[:, 27, 2], label="ankle z (raw)")
plt.plot(norm_pose[:, 27, 2], label="ankle z (normalized)")
plt.legend()

# pelvis should be near zero
np.mean(pose_scaled[:, [23, 24]], axis=(0,1))

# scale consistency
np.linalg.norm(pose_scaled[:, 11] - pose_scaled[:, 23], axis=1).mean()


In [ ]:
# 3. Joint Selection - Research has shown that being selective about joints can improve outcomes because it reduces noise. 
# Given our limited number of output anomalies that we want to predict (due to limited input data), we focus on the following joints, most relevant for gait analysis:
# Store as joints.py ??

# A. Define joint indices

GAIT_JOINTS = [
    2, 5,     # eyes (head orientation)
    11, 12,   # shoulders
    23, 24,   # hips
    25, 26,   # knees
    27, 28,   # ankles
    29, 30,   # heels
    31, 32    # foot index
]

JOINT_NAMES = {
    2:  "left_eye",
    5:  "right_eye",

    11: "left_shoulder",
    12: "right_shoulder",

    23: "left_hip",
    24: "right_hip",

    25: "left_knee",
    26: "right_knee",

    27: "left_ankle",
    28: "right_ankle",

    29: "left_heel",
    30: "right_heel",

    31: "left_foot_index",
    32: "right_foot_index"
}

GAIT_JOINT_GROUPS = {
    "head": {
        2: "left_eye",
        5: "right_eye"
    },

    "trunk": {
        11: "left_shoulder",
        12: "right_shoulder",
        23: "left_hip",
        24: "right_hip"
    },

    "lower_limbs": {
        25: "left_knee",
        26: "right_knee",
        27: "left_ankle",
        28: "right_ankle",
        29: "left_heel",
        30: "right_heel",
        31: "left_foot_index",
        32: "right_foot_index"
    }
}

# B. Joint selection function 

def select_gait_joints(pose, joint_indices):
    pose = np.asarray(pose)
    assert pose.ndim == 3 and pose.shape[1] >= max(joint_indices) + 1
    return pose[:, joint_indices, :]


In [ ]:
# 4. Detect gait cycles automatically based on heel strike (FPS aware)

LEFT_HEEL, RIGHT_HEEL = 29, 30

def find_heel_strikes(X, foot='left', fps=30, min_time_between_steps=0.5, smooth_sigma=1):
    """
    Detect heel strikes for one foot using y-coordinate minima.
    
    X: (T, J, 3)
    foot: 'left' or 'right'
    fps: frames per second of this clip
    min_time_between_steps: minimum seconds between consecutive heel strikes
    smooth_sigma: smoothing of heel y-coordinate
    """
    heel_idx = LEFT_HEEL if foot=='left' else RIGHT_HEEL
    y = X[:, heel_idx, 1]
    
    # Smooth
    y_smooth = gaussian_filter1d(y, sigma=smooth_sigma)
    
    # Convert time to frames
    min_distance_frames = int(min_time_between_steps * fps)
    
    peaks, _ = find_peaks(-y_smooth, distance=min_distance_frames)
    return peaks


In [ ]:
# 5 Extract Gait Cycle Clips

def extract_gait_cycle_clips(pose, fps, cycles=1, min_time_between_steps=0.5,
                             min_frames=40, max_frames=150, resample_frames=60):
    """
    Extract gait-cycle aligned clips from pose data, automatically using clip FPS.
    
    X: (T, J, 3)
    fps: frames per second
    cycles: number of gait cycles per clip
    min_time_between_steps: seconds between heel strikes
    """

    """
    Returns list of clips: (resample_frames, 33, 3)
    """
    clips = []
    
    # Detect heel strikes
    left_events = find_heel_strikes(pose, "left", fps, min_time_between_steps)
    right_events = find_heel_strikes(pose, "right", fps, min_time_between_steps)

    for foot_events in [left_events, right_events]:
        for i in range(len(foot_events) - cycles):
            s = foot_events[i]
            e = foot_events[i + cycles]
            clip = pose[s:e]

            if min_frames <= len(clip) <= max_frames:
                clip = resample(clip, resample_frames, axis=0)
                clips.append(clip)
                
    return clips


In [ ]:
#6. Label Mapping

LABEL_MAP = {
    "normal gait": 0,
    "abnormal gait": 1
}

In [ ]:
#7 Full pipeline for dataframe - CHECK / NOT FINAL
# ---------------------------
def preprocess_gait_dataframe(df_video, cycles=1, resample_frames=60):
    """
    df_video must contain:
      - pose: (T,33,3)
      - fps
      - dataset (label)
    """

    all_clips = []
    all_labels = []

    for _, row in df_video.iterrows():
        pose = row["pose"]
        fps = row["fps"]
        label = LABEL_MAP[row["dataset"]]

        pose_norm = normalize_pose_3d(pose)

        pose_norm = select_gait_joints(pose, GAIT_JOINTS)

        clips = extract_gait_cycle_clips(
            pose_norm,
            fps=fps,
            cycles=cycles,
            resample_frames=resample_frames
        )

        all_clips.extend(clips)
        all_labels.extend([label] * len(clips))

    X = np.array(all_clips)    # (N_clips, T, 33, 3)
    y = np.array(all_labels)   # (N_clips,)

    return X, y


In [ ]:
#8. Use Example

# # Step 1: Convert raw dataframe
df_video = add_pose_column(df_raw)

# Step 2: Preprocess for ML
X_clips, y_labels = preprocess_gait_dataframe(
    df_video,
    cycles=1,
    resample_frames=60
)

print(X_clips.shape)  # (N_clips, 60, 33, 3)
print(y_labels.shape)

In [ ]:
# READY for both approaches

#Feature-based ML (XGBoost / RF)
# Example reshape for feature extraction
clip = X_clips[0]  # (60, 33, 3)

# Deep Learning
X_dl = X_clips.reshape(len(X_clips), 60, -1)


We utilized the 3D output of MediaPipe Pose (x, y, z), where the z-dimension represents relative depth. All joints were pelvis-centered and scaled by torso length to ensure comparability across subjects.

In [ ]:
                ┌───────────────┐
30fps healthy → │ Pose sequence │
60fps pathology │  (T, J, 3)    │
                └──────┬────────┘
                       │
                       ▼
           ┌─────────────────────────┐
           │  Normalize Pose 3D      │
           │  - Pelvis-centered      │
           │  - Scaled by torso      │
           │  - Works for x, y, z   │
           └─────────┬──────────────┘
                     │
                     ▼
           ┌─────────────────────────┐
           │ Detect Heel Strikes      │
           │  - Use FPS from metadata │
           │  - Convert min_time → frames
           │  - Separate left/right │
           └─────────┬──────────────┘
                     │
                     ▼
           ┌─────────────────────────┐
           │ Extract Gait-Cycle Clips│
           │  - Each clip = 1–2 cycles│
           │  - Clips may vary in length│
           │  - Filter min/max frames │
           └─────────┬──────────────┘
                     │
                     ▼
           ┌─────────────────────────┐
           │ Resample Clips           │
           │  - Fixed number of frames│
           │  - Example: 60 frames   │
           │  - Works for 30fps & 60fps│
           └─────────┬──────────────┘
                     │
                     ▼
           ┌─────────────────────────┐
           │ Optional: Feature Calc  │
           │  - x, y, z coordinates  │
           │  - Velocity, acceleration│
           │  - Normalize by dt=1/fps│
           └─────────┬──────────────┘
                     │
                     ▼
           ┌─────────────────────────┐
           │ ML Model Input           │
           │  - Shape: (resample_frames, J*3) │
           │  - LSTM / Transformer / CNN │
           └─────────────────────────┘
